In [7]:
import os 
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from dotenv import load_dotenv
import yaml
import matplotlib.pyplot as plt
import seaborn as sns 

import warnings
warnings.filterwarnings("ignore")

In [8]:
spark = (SparkSession.builder                                                                                                                                                       
    .master("local[*]")
    .appName("SerumQA-feature-engineering")
    .config("spark.sql.shuffle.partitions", "8")                                                                                                                               
    .getOrCreate()
) 

In [9]:
                                                                                                                                                                     
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/rushikesh/Music/drivers/postgresql-42.7.3.jar pyspark-shell'

In [10]:
load_dotenv()

BASE_DIR = Path("..").resolve()
CONFIG_PATH = BASE_DIR / "configs" / "config.yaml"

In [11]:
with open(CONFIG_PATH) as f:                                                                                                                                                        
    config = yaml.safe_load(f)

In [12]:
spark.stop()

spark = (SparkSession.builder
         .master("local[*]")
         .appName("SerumQA-feature-engineering")
         .config("spark.jars", str(BASE_DIR / "drivers" / "postgresql-42.7.3.jar"))
         .config("spark.driver.extraClassPath", str(BASE_DIR / "drivers" / "postgresql-42.7.3.jar"))
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate()
)


In [13]:
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.1.1


In [14]:
jdbc_url = (
        f"jdbc:postgresql://"
        f"{os.getenv('POSTGRES_HOST', 'localhost')}:"
        f"{os.getenv('POSTGRES_PORT', '5433')}/"
        f"{os.getenv('POSTGRES_DB')}"
    )

In [15]:

properties = {
    "user": os.getenv("POSTGRES_USER"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

In [16]:
silver_df = spark.read.jdbc(jdbc_url, "silver.qa_events_clean", properties=properties)

In [18]:
print(f"Silver row count: {silver_df.count():,}")

silver_df.printSchema()

Silver row count: 431,068
root
 |-- id: integer (nullable = true)
 |-- bronze_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- system: string (nullable = true)
 |-- root_cause: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_days: double (nullable = true)
 |-- closed_timestamp: timestamp (nullable = true)
 |-- is_anomaly: integer (nullable = true)
 |-- is_covid_period: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- cleaned_at: timestamp (nullable = true)



In [19]:
silver_df.select("system", "event_type", "severity", "is_anomaly").show(5)

+-------+----------+--------+----------+
| system|event_type|severity|is_anomaly|
+-------+----------+--------+----------+
| QC Lab|       ade|   Minor|         0|
| QC Lab|       ade|   Major|         0|
| QC Lab|       ade|   Minor|         0|
| QC Lab|       ade|   Major|         0|
|Unknown|       ade|   Minor|         0|
+-------+----------+--------+----------+
only showing top 5 rows


In [20]:
print("Systems:", [r[0] for r in silver_df.select("system").distinct().collect()])

Systems: ['Filling', 'Warehouse', 'Manufacturing', 'Blending', 'Documentation', 'QC Lab', 'Unknown', 'Capping', 'Lyophilization']


In [21]:
total = silver_df.count()

anomalies = silver_df.filter(F.col("is_anomaly") == 1).count()

print(f"Total : {total:,}")
print(f"Anomalies: {anomalies:,}")
print(f"Anomaly rate: {anomalies/total:.4%}")

Total : 431,068
Anomalies: 2,006
Anomaly rate: 0.4654%


In [22]:
days = lambda d: d * 86400


window_7d = (
    Window.partitionBy("system")
    .orderBy(F.col("timestamp").cast("long"))
    .rangeBetween(-days(7), 0)
)

window_30d = (
    Window.partitionBy("system")
    .orderBy(F.col("timestamp").cast("long"))
    .rangeBetween(-days(30), 0)
)


In [23]:
gold_df = silver_df.withColumn("event_count_7d", F.count("*").over(window_7d)) \
    .withColumn("event_count_30d", F.count("*").over(window_30d)) 

gold_df.select("system", "timestamp", "event_count_7d", "event_count_30d").show(5)


+--------+-------------------+--------------+---------------+
|  system|          timestamp|event_count_7d|event_count_30d|
+--------+-------------------+--------------+---------------+
|Blending|2020-05-15 14:16:10|             1|              1|
|Blending|2020-05-16 11:55:27|             2|              2|
|Blending|2020-05-16 14:44:08|             3|              3|
|Blending|2020-05-16 21:44:42|             4|              4|
|Blending|2020-05-18 07:45:32|             5|              5|
+--------+-------------------+--------------+---------------+
only showing top 5 rows


In [26]:
gold_df = gold_df.withColumn(
    "anomaly_count_7d", F.sum("is_anomaly").over(window_7d)).withColumn(
    "anomaly_count_30d", F.sum("is_anomaly").over(window_30d)).withColumn(
    "is_deviation", (F.col("event_type") == "deviation").cast("int")).withColumn(
    "deviation_count_7d", F.sum("is_deviation").over(window_7d)).withColumn(
    "deviation_count_30d", F.sum("is_deviation").over(window_30d))

gold_df.select("timestamp", "system", "anomaly_count_7d", "deviation_count_7d").show(5)

+-------------------+--------+----------------+------------------+
|          timestamp|  system|anomaly_count_7d|deviation_count_7d|
+-------------------+--------+----------------+------------------+
|2020-05-15 14:16:10|Blending|               0|                 0|
|2020-05-16 11:55:27|Blending|               0|                 0|
|2020-05-16 14:44:08|Blending|               0|                 0|
|2020-05-16 21:44:42|Blending|               0|                 0|
|2020-05-18 07:45:32|Blending|               0|                 0|
+-------------------+--------+----------------+------------------+
only showing top 5 rows


In [27]:
gold_df = gold_df.withColumn(
    "is_oos", (F.col("event_type") == "oos").cast("int")).withColumn(
    "oos_count_7d", F.sum("is_oos").over(window_7d)).withColumn(
    "oos_count_30d", F.sum("is_oos").over(window_30d)).withColumn(
    "is_critical", (F.col("severity") == "critical").cast("int")).withColumn(
    "critical_ratio_7d", F.sum("is_critical").over(window_7d) / F.count("*").over(window_7d)).withColumn(
    "critical_ratio_30d", F.sum("is_critical").over(window_30d) / F.count("*").over(window_30d)
    )

gold_df.select("timestamp", "system", "oos_count_7d", "critical_ratio_7d").show(5)

+-------------------+--------+------------+-----------------+
|          timestamp|  system|oos_count_7d|critical_ratio_7d|
+-------------------+--------+------------+-----------------+
|2020-05-15 14:16:10|Blending|           0|              0.0|
|2020-05-16 11:55:27|Blending|           0|              0.0|
|2020-05-16 14:44:08|Blending|           0|              0.0|
|2020-05-16 21:44:42|Blending|           0|              0.0|
|2020-05-18 07:45:32|Blending|           0|              0.0|
+-------------------+--------+------------+-----------------+
only showing top 5 rows


In [28]:
gold_df = gold_df.withColumn("anomaly_rate_7d", F.col("anomaly_count_7d") / F.col("event_count_7d")).withColumn(
    "anomaly_rate_30d", F.col("anomaly_count_30d") / F.col("event_count_30d"))

gold_df.select("timestamp", "system", "event_count_7d", "anomaly_rate_7d", "anomaly_count_7d").show(5)

+-------------------+--------+--------------+---------------+----------------+
|          timestamp|  system|event_count_7d|anomaly_rate_7d|anomaly_count_7d|
+-------------------+--------+--------------+---------------+----------------+
|2020-05-15 14:16:10|Blending|             1|            0.0|               0|
|2020-05-16 11:55:27|Blending|             2|            0.0|               0|
|2020-05-16 14:44:08|Blending|             3|            0.0|               0|
|2020-05-16 21:44:42|Blending|             4|            0.0|               0|
|2020-05-18 07:45:32|Blending|             5|            0.0|               0|
+-------------------+--------+--------------+---------------+----------------+
only showing top 5 rows


In [30]:
gold_df.filter(F.col("anomaly_rate_7d") > 0).select(
    "timestamp", "system", "anomaly_rate_7d", "event_count_7d", "anomaly_count_7d").orderBy(F.col("anomaly_rate_7d").desc()).show(10)

+-------------------+-------+------------------+--------------+----------------+
|          timestamp| system|   anomaly_rate_7d|event_count_7d|anomaly_count_7d|
+-------------------+-------+------------------+--------------+----------------+
|2020-05-18 23:14:43|Filling|0.9621212121212122|           132|             127|
|2020-05-18 22:47:16|Filling|0.9618320610687023|           131|             126|
|2020-05-18 22:36:57|Filling|0.9615384615384616|           130|             125|
|2020-05-18 22:36:15|Filling|0.9612403100775194|           129|             124|
|2020-05-18 22:31:18|Filling|         0.9609375|           128|             123|
|2020-05-18 22:13:21|Filling|0.9606299212598425|           127|             122|
|2020-05-18 21:57:49|Filling|0.9603174603174603|           126|             121|
|2020-05-18 21:55:38|Filling|              0.96|           125|             120|
|2020-05-18 21:49:30|Filling|0.9596774193548387|           124|             119|
|2020-05-18 21:30:33|Filling

In [35]:
feature_cols = [
    "id", "timestamp", "system", "event_type", "severity", "is_anomaly", 
    "event_count_7d", "event_count_30d", "anomaly_count_7d", "anomaly_count_30d",
    "oos_count_7d", "oos_count_30d", "critical_ratio_7d", "critical_ratio_30d",
    "anomaly_rate_7d", "anomaly_rate_30d",
    "deviation_count_7d", "deviation_count_30d"
]

try:
    gold_df.select(feature_cols).write.jdbc(
        url=jdbc_url,
        table="gold.qa_features",
        mode="overwrite",
        properties=properties
    )
except Exception as e:
    print(str(e))

In [36]:
verify_df = spark.read.jdbc(jdbc_url, "gold.qa_features", properties=properties)
print(f"Gold row count: {verify_df.count():,}")

Gold row count: 431,068
